<a href="https://colab.research.google.com/github/HarshvardhanJ/FlashAttention/blob/main/FlashAttention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# 1. Forward Pass

I am going to break the computation into chunks by utilizing online softmax.

$$attention(q,k,v) = \frac{∑_iv_i\cdot s'_i}{∑_js'_j}$$ where $$s'_i = e^{q\cdot k_i}$$

In [ ]:
def onlineSoftmax(max_old, den_old, sum_old, scores_curr, values_curr):
    # here max_old, den_old, sum_old are the previous maximum, denominator and accumulated weighted sum values,
    # and scores_curr and values_curr are the current best scores and its corresponding vectors
    max_curr_chunk = scores_curr.max()
    max_new = max(max_old, max_curr_chunk)

    scale = np.exp(max_old-max_new) # rescaling for numerical stability
    scaled_scores = np.exp(scores_curr-max_new)

    den_new = den_old * scale + scaled_scores.sum()
    sum_new = sum_old * scale + scaled_scores @ values_curr

    return max_new, den_new, sum_new


In [ ]:
# X - (n, d)
# chunk - (C, d)
# score - O(C·d)
def attentionForward(X, C=16):
    # C is the chunk size
    n, d = X.shape
    max_curr = np.full(n, float('-inf'))
    den_curr = np.zeros(n)
    sum_curr = np.zeros((n,d))

    for j_start in range(0, n, C):
        j_end = min(j_start + C, n)
        chunk = X[j_start:j_end]

        scores = X @ chunk.T

        for i in range(n):
            max_curr[i], den_curr[i], sum_curr[i] = onlineSoftmax(max_curr[i], den_curr[i], sum_curr[i], scores[i], chunk)

    Y = sum_curr / den_curr[:, None]
    return Y, max_curr, den_curr

# Backward Pass

In [ ]:
def attentionBackward(dY, X, Y, m, den, C=16):
    n, d = X.shape
    D = (dY * Y).sum(axis=1)
    dX = np.zeros((n,d))

    for j_start in range(0, n, C):
        j_end = min(j_start+C, n)
        chunk = X[j_start:j_end]
        scores = X @ chunk.T
        Prob_chunk = np.exp(scores - m[:, None])/den[:, None]

        dX[j_start:j_end] += Prob_chunk.T @ dY

        dProb_chunk = dY @ chunk.T
        dScore_chunk = Prob_chunk * (dProb_chunk - D[:, None])

        dX += dScore_chunk @ chunk
        dX[j_start:j_end] += dScore_chunk.T @ X

    return dX

# Testing

!! The following block of code is LLM generated just for summarizing the results.

In [ ]:
rng = np.random.default_rng(42)
X = rng.standard_normal((128, 32))
n, d = X.shape

Y, m, den = attentionForward(X, C=16)
print("Y shape :", Y.shape)

dY = np.ones_like(Y)
dX = attentionBackward(dY, X, Y, m, den, C=16)
print("dX shape:", dX.shape)

Y shape : (128, 32)
dX shape: (128, 32)


```

  Space Complexity  |  n=128, d=32, C=16


  FORWARD PASS
  Tensor                 Formula    Floats                   Lifecycle
  ------------------------------------------------------------
  X  (input)             n·d             4,096  (  16.0 KB)  persistent — input
  scores scratch         n·C             2,048  (   8.0 KB)  temporary — reused each chunk
  chunk scratch          C·d               512  (   2.0 KB)  temporary — reused each chunk
  m  (running max)       n                 128  (   0.5 KB)  persistent — saved for backward
  den (denom)            n                 128  (   0.5 KB)  persistent — saved for backward
  Y  (output)            n·d             4,096  (  16.0 KB)  persistent — saved for backward

  Peak during loop    O(n·d + n·C)  =       6,656  (  26.0 KB)
  Persisted after     O(n·d)        =       8,448  (  33.0 KB)

  BACKWARD PASS
  Tensor                 Formula    Floats                   Lifecycle
  ------------------------------------------------------------
  dY (upstream)          n·d             4,096  (  16.0 KB)  persistent — given from loss
  D  = dot(dY,Y)         n                 128  (   0.5 KB)  persistent — O(n·d) to compute, O(n) to store
  dX (gradient)          n·d             4,096  (  16.0 KB)  persistent — accumulated output
  Prob_chunk             n·C             2,048  (   8.0 KB)  temporary — recomputed each chunk
  dProb_chunk            n·C             2,048  (   8.0 KB)  temporary — scratch each chunk
  dScore_chunk           n·C             2,048  (   8.0 KB)  temporary — scratch each chunk

  Peak during loop    O(n·d + n·C)  =      14,464  (  56.5 KB)

  ==========================================================
  Pass          Peak floats     Peak KB  Big-O
  ----------------------------------------------------------
  Forward             6,656       26.0  O(n·d + n·C)
  Backward           14,464       56.5  O(n·d + n·C)
  ──────────────────────────────────────────────────────────
  Since C is a fixed constant → both passes are O(n·d)
  ```

In [ ]:
# The vanilla attention uses O(n^2) space complexity for both forward and backward pass, while this approach is have O(n*d) which is significantly better.

# 1. Self-attention Does Not Need O(n^2) Memory -> https://arxiv.org/pdf/2112.05682

# 2. FlashAttention Fast and Memory-Efficient Exact Attention with IO-Awareness-> https://arxiv.org/pdf/2205.14135